In [1]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.8 MB/s eta 0:00:00


In [2]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from huggingface_hub import notebook_login

# 1. Login to Hugging Face to access Llama 3
notebook_login()

# 2. Choose the Model
# Change it to the Llama 3.2 model which you are already approved for!
model_id = "Qwen/Qwen2.5-7B-Instruct"



# 3. Load the combined dataset we just created
dataset = load_dataset('json', data_files='prepared_training_data.jsonl')

# 4. Setup 4-bit Quantization (Must-have for Colab)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 5. Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 6. Apply LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)


# 7. Training Arguments (Updated for trl version 1.2.0+)
training_args = SFTConfig(
    output_dir="./crime_model_outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=300,
    optim="paged_adamw_8bit",
    dataset_text_field="text", # Back inside SFTConfig
    max_length=512,            # Renamed from max_seq_length!
)

# 8. Train the Model
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    peft_config=peft_config,
    args=training_args,
)

print("Starting training on your combined CSV data...")
trainer.train()

# 9. Save your model
trainer.model.save_pretrained("my-finetuned-second-crime-model")
tokenizer.save_pretrained("my-finetuned-second-crime-model")
print("All done!")




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/213830 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training on your combined CSV data...


Step,Training Loss
10,2.561652
20,0.856078
30,0.453561
40,0.378548
50,0.355520
60,0.346567
70,0.336059
80,0.331736
90,0.340832
100,0.350843


All done!


In [3]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Setup the 4-bit config (The new required way)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Load the base model and tokenizer
base_model_id ="Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    quantization_config=bnb_config  # <-- Using the config here instead!
)

# 3. Attach your newly trained LoRA weights to it
model = PeftModel.from_pretrained(base_model, "my-finetuned-second-crime-model")

# 4. Ask it a question!
question = "What type of crime occurred on or near Dunbar Gardens in 2025-12?"
prompt = f"### User: {question}\n### Assistant:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("Generating answer...\n")
# 5. Generate the response
outputs = model.generate(**inputs, max_new_tokens=60, temperature=0.3)

# 6. Print the final answer nicely
raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
final_answer = raw_output.split("### Assistant:")[-1].strip()

print("--- MODEL ANSWER ---")
print(final_answer)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Generating answer...

--- MODEL ANSWER ---
In 2025-12, a violence and sexual offences was reported on or near dunbar gardens. The current status is: Unable to prosecute suspect.


In [4]:
# Replace 'Your_HuggingFace_Username' with your actual username
hf_repo_name = "VasuReddy07/Qwen-2.5-Crime-QA"

print(f"Uploading model to {hf_repo_name}...")

# 1. Push the LoRA Adapter Weights
model.push_to_hub(hf_repo_name)

# 2. Push the Tokenizer
tokenizer.push_to_hub(hf_repo_name)

print("Upload complete! You can now view your model on the Hugging Face website.")


Uploading model to VasuReddy07/Qwen-2.5-Crime-QA...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   3%|3         | 1.21MB / 40.4MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpufkqhjn0/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Upload complete! You can now view your model on the Hugging Face website.


In [5]:
import shutil
from google.colab import files

print("Zipping the model files...")
# This compresses the model folder into a single ZIP file
shutil.make_archive('crime_lora_second_model', 'zip', 'my-finetuned-second-crime-model')

print("Starting download to your local machine! Please wait...")
# This triggers your browser to download the file
files.download('crime_lora_second_model.zip')


Zipping the model files...
Starting download to your local machine! Please wait...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
!pip install -q -U transformers peft accelerate bitsandbytes huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 47.3 MB/s eta 0:00:00


In [2]:
from huggingface_hub import notebook_login
notebook_login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Configuration
hf_model_path = "VasuReddy07/Qwen-2.5-Crime-QA"
base_model_id = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Load Tokenizer & Base Model
print("Downloading base model...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    quantization_config=bnb_config
)

# 3. Attach YOUR fine-tuned weights!
print(f"Downloading your custom adapter from {hf_model_path}...")
model = PeftModel.from_pretrained(model, hf_model_path)

print("Model successfully loaded into memory!")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/40.4M [00:00<?, ?B/s]

Model successfully loaded into memory!


In [4]:
def ask_odin(query: str, max_tokens: int = 128, temperature: float = 0.3) -> str:
    messages = [
        {
            "role": "system",
            "content": """You are ODIN (Operational Decision Intelligence Network), an AI assistant
specialized in UK policing and crime analysis. You provide accurate, professional, and concise responses
based on police reports and location data."""
        },
        {"role": "user", "content": query}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print("ODIN is ready!")


ODIN is ready!


In [5]:
# Ask a question based on the CSV data it learned
test_query = "What type of crime occurred on or near Dunbar Gardens in 2025-12?"
print(f"[User Query]: {test_query}\n" + "-"*60)
print(f"[ODIN Response]:\n{ask_odin(test_query)}")


[User Query]: What type of crime occurred on or near Dunbar Gardens in 2025-12?
------------------------------------------------------------
[ODIN Response]:
In 2025-12, a violence and sexual offences was reported on or near dunbar gardens. The current status is: Unable to prosecute suspect.


In [6]:
test_queries = [
    "What are effective strategies for reducing burglary in residential areas?",
    "Explain the OSARA framework for operational planning.",
    "What does the Theft Act 1968 say about robbery?",
    "How can predictive policing help with resource allocation?",
    "What are evidence-based approaches to reducing youth violence?"
]

print("="*70)
print("ODIN Fine-Tuned Model - Policing Query Demo")
print("="*70)

# We use a loop to send each query to ODIN one at a time
for i, query in enumerate(test_queries, 1):
    print(f"\n[User Query {i}]: {query}")
    print("-" * 60)

    # Send the single query to the function
    response = ask_odin(query, max_tokens=256)

    print(f"[ODIN Response]:\n{response}")
    print("=" * 70)


ODIN Fine-Tuned Model - Policing Query Demo

[User Query 1]: What are effective strategies for reducing burglary in residential areas?
------------------------------------------------------------
[ODIN Response]:
Reducing burglary in residential areas can be approached through a variety of strategies. Here are some effective methods:

1. **Community Policing**: Engage with local communities to build trust and encourage reporting of suspicious activities. Regular meetings between police and residents can help identify and address issues early.

2. **Neighborhood Watch Programs**: Encourage the formation of neighborhood watch groups. These programs allow residents to monitor their area and report crimes or suspicious behavior promptly.

3. **Public Lighting**: Improve street lighting in residential areas to deter potential burglars. Well-lit areas can make it more difficult for criminals to operate unnoticed.

4. **Surveillance Cameras**: Install CCTV cameras in high-risk areas. Public a